# 🔒 ISOM 260: Three Things Your Bot Can't Do Yet

Last Wednesday your agent moved onto your phone. Your friends texted it. Some of you had a parent text it.

It also has three holes a business would never accept:

| # | The hole | What a company calls it | Today's fix |
|---|---|---|---|
| 1 | **Anyone** with the link can use it — and burn your API quota | authentication | an allow-list: a door with your name on it |
| 2 | It **acts without asking** — if it had a real `send_email`, it would send | human-in-the-loop | an approval gate *inside the chat*: 🚦 reply YES |
| 3 | It **forgets everything** the moment the cell stops | persistence | memory that lives on disk — and a `/forget` command |

Plus one bonus: a **real-world API** (weather) that fails in the interesting way — not with an error, but with a confident wrong answer.

Everything from last week is still here. The new lines are marked `NEW`.

## 🚀 Part 1: Setup (2 minutes)

Same secrets as last week: `GOOGLE_API_KEY`, `TELEGRAM_BOT_TOKEN`, and optionally the two Adzuna keys. Nothing new to create.

In [ ]:
!pip install -q google-genai

In [ ]:
from google.colab import userdata

try:
    GOOGLE_API_KEY     = userdata.get("GOOGLE_API_KEY")
    TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")
    print("✅ Both secrets loaded from Colab Secrets.")
except Exception:
    raise SystemExit("❌ Missing a secret. Open the 🔑 panel, add GOOGLE_API_KEY and TELEGRAM_BOT_TOKEN "
                     "(the token from last week — BotFather → /mybots → API Token if you lost it), switch on Notebook access.")

try:
    ADZUNA_APP_ID, ADZUNA_APP_KEY = userdata.get("ADZUNA_APP_ID"), userdata.get("ADZUNA_APP_KEY")
    print("✅ Adzuna key found — local job search is live.")
except Exception:
    ADZUNA_APP_ID = ADZUNA_APP_KEY = None
    print("ℹ️  No Adzuna key — local job search falls back to remote listings (and says so).")

In [ ]:
from google import genai
from google.genai import types, errors
import urllib.request, urllib.parse, json, time, datetime, re

client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL  = "gemini-2.5-flash"

def tg(method, http_timeout=20, **params):
    """One function for the whole Telegram Bot API: tg('sendMessage', chat_id=..., text=...)"""
    req = urllib.request.Request(f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/{method}",
                                 data=json.dumps(params).encode(),
                                 headers={"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=http_timeout))["result"]

me = tg("getMe")
BOT_LINK = f"https://t.me/{me['username']}"
print(f"🧠 Gemini ready · 🤖 @{me['username']} → {BOT_LINK}")

## 🛠️ Part 2: Tools — last week's four, plus two

New: **`get_weather`**, a real public API with no key, and **`send_email`**, a tool that *acts*. It is simulated — nothing is sent — but the agent does not know that, which is exactly the point of today.

Run the cell, then look at the second test line. You asked for Boston. Read what the service *matched*.

In [ ]:
# ============================================
# TOOLS — the same shape as Session 4: a job description + a Python function
# ============================================

# Tool 1: the calculator you already know
calculator_tool = types.FunctionDeclaration(
    name="calculator",
    description=(
        "A precise calculator. Use it for ANY arithmetic: addition, subtraction, "
        "multiplication, division, exponentiation. Always prefer it over mental math."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"expression": {"type": "string", "description": "e.g. '(100 * 0.15) + 50'"}},
        "required": ["expression"],
    },
)

def calculator(expression: str) -> str:
    if not all(c in set("0123456789+-*/.() ") for c in expression):
        return "Error: only numbers and + - * / . ( ) are allowed."
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"


# Tool 2: a REAL tool — live remote job listings (public API, no key needed)
jobs_tool = types.FunctionDeclaration(
    name="find_jobs",
    description=(
        "Search live remote job listings by a keyword such as 'marketing', 'analyst', "
        "'finance', 'sales' or 'design'. Returns up to 5 current openings with company, "
        "salary (when listed) and a link. Use it whenever the user asks about jobs or careers."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"keyword": {"type": "string", "description": "One or two words, e.g. 'marketing'"}},
        "required": ["keyword"],
    },
)

def find_jobs(keyword: str, limit: int = 5) -> str:
    req = urllib.request.Request("https://remoteok.com/api",
                                 headers={"User-Agent": "ISOM260-student-project"})
    raw = json.load(urllib.request.urlopen(req, timeout=25))
    pool = [j for j in raw if isinstance(j, dict) and j.get("position")]
    kw = keyword.lower()
    hits = [j for j in pool if kw in j.get("position", "").lower()]                # in the title
    hits += [j for j in pool if j not in hits and kw in " ".join(j.get("tags", [])).lower()]  # in the tags
    jobs = []
    for j in hits[:limit]:
        sal = f"${j['salary_min']:,}-${j.get('salary_max', j['salary_min']):,}" if j.get("salary_min") else "not listed"
        jobs.append({"title": j["position"], "company": j.get("company", "?"),
                     "location": j.get("location") or "Remote", "salary": sal, "url": j.get("url", "")})
    if not jobs:
        return json.dumps({"result": f"No live listings match '{keyword}'. Suggest a broader keyword."})
    return json.dumps(jobs)


# Tool 3: LOCAL jobs via Adzuna (optional key) — falls back to the remote board, and says so
adzuna_tool = types.FunctionDeclaration(
    name="find_local_jobs",
    description=(
        "Search real job postings near a US city (default Boston) by keyword, e.g. 'business analyst'. "
        "Returns up to 5 openings with company, location, salary and a link, plus a 'source' field. "
        "If source says FALLBACK, the listings are REMOTE jobs, not local — tell the user that plainly."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"keyword":  {"type": "string", "description": "Job title or skill, e.g. 'financial analyst'"},
                       "location": {"type": "string", "description": "City, e.g. 'Boston'. Default Boston."}},
        "required": ["keyword"],
    },
)

def find_local_jobs(keyword: str, location: str = "Boston", limit: int = 5) -> str:
    def fallback(why):
        return json.dumps({"source": f"FALLBACK ({why}) — these are REMOTE listings, not jobs in {location}",
                           "jobs": json.loads(find_jobs(keyword, limit))})
    if not (ADZUNA_APP_ID and ADZUNA_APP_KEY):
        return fallback("no Adzuna key")
    try:
        q = urllib.parse.urlencode({"app_id": ADZUNA_APP_ID, "app_key": ADZUNA_APP_KEY, "what": keyword,
                                    "where": location, "results_per_page": limit, "content-type": "application/json"})
        req = urllib.request.Request(f"https://api.adzuna.com/v1/api/jobs/us/search/1?{q}",
                                     headers={"User-Agent": "ISOM260-student-project"})
        raw = json.load(urllib.request.urlopen(req, timeout=20))
        jobs = []
        for r in raw.get("results", [])[:limit]:
            sal = f"${int(r['salary_min']):,}-${int(r.get('salary_max') or r['salary_min']):,}" if r.get("salary_min") else "not listed"
            jobs.append({"title": r.get("title", "?"), "company": (r.get("company") or {}).get("display_name", "?"),
                         "location": (r.get("location") or {}).get("display_name", location), "salary": sal,
                         "url": r.get("redirect_url", ""), "snippet": " ".join((r.get("description") or "").split())[:200]})
        return json.dumps({"source": f"Adzuna live · {raw.get('count', len(jobs))} postings match near {location}", "jobs": jobs})
    except Exception as e:
        return fallback(f"Adzuna failed: {type(e).__name__}")


# Tool 4: YOUR HW #5 tool goes here (paste the declaration + function, then register it below)


TOOLS     = [calculator_tool, jobs_tool, adzuna_tool]                     # ← add your tool's declaration
FUNCTIONS = {"calculator": calculator, "find_jobs": find_jobs, "find_local_jobs": find_local_jobs}   # ← and its function


# ============================================
# NEW TODAY — two more tools
# ============================================

# Tool 5: a REAL-WORLD API with real failure modes — weather, no key (Open-Meteo)
weather_tool = types.FunctionDeclaration(
    name="get_weather",
    description=(
        "Current weather and a 2-day outlook for a city. Returns the place the service actually matched "
        "(check it — a misspelled city can match the wrong country), temperatures in °F, wind, and rain chance."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"city": {"type": "string", "description": "City name, e.g. 'Boston'"}},
        "required": ["city"],
    },
)

def get_weather(city: str) -> str:
    q = urllib.parse.urlencode({"name": city, "count": 1, "language": "en"})
    geo = json.load(urllib.request.urlopen(f"https://geocoding-api.open-meteo.com/v1/search?{q}", timeout=15))
    if not geo.get("results"):
        return f"Error: no place called '{city}' was found. Ask the user to check the spelling."
    g = geo["results"][0]
    q2 = urllib.parse.urlencode({"latitude": g["latitude"], "longitude": g["longitude"],
                                 "current": "temperature_2m,wind_speed_10m", "temperature_unit": "fahrenheit",
                                 "wind_speed_unit": "mph", "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max",
                                 "forecast_days": 2, "timezone": "auto"})
    w = json.load(urllib.request.urlopen(f"https://api.open-meteo.com/v1/forecast?{q2}", timeout=15))
    return json.dumps({"matched_place": f"{g['name']}, {g.get('admin1', '')}, {g['country']}",
                       "now_f": w["current"]["temperature_2m"], "wind_mph": w["current"]["wind_speed_10m"],
                       "today_high_f": w["daily"]["temperature_2m_max"][0], "today_low_f": w["daily"]["temperature_2m_min"][0],
                       "rain_chance_pct": w["daily"]["precipitation_probability_max"][0],
                       "tomorrow_high_f": w["daily"]["temperature_2m_max"][1]})


# Tool 6: a tool that ACTS — simulated, but the agent doesn't know that
email_tool = types.FunctionDeclaration(
    name="send_email",
    description="Send an email on the user's behalf. Use ONLY when the user explicitly asks to send or email something.",
    parameters_json_schema={
        "type": "object",
        "properties": {"to": {"type": "string"}, "subject": {"type": "string"}, "body": {"type": "string"}},
        "required": ["to", "subject", "body"],
    },
)

def send_email(to: str, subject: str, body: str) -> str:
    return f"SENT (simulated) to {to} · subject '{subject}' · {len(body)} characters"


TOOLS     += [weather_tool, email_tool]
FUNCTIONS.update({"get_weather": get_weather, "send_email": send_email})
RISKY = {"send_email"}        # ← tools that must not run without a human saying yes

print(f"🛠️  {len(TOOLS)} tools: {', '.join(FUNCTIONS)}  · risky: {RISKY}")
print("   get_weather('Boston') →", json.loads(get_weather("Boston"))["matched_place"])
print("   get_weather('Bostn')  →", json.loads(get_weather("Bostn"))["matched_place"], "   ← a typo, and the API did not complain")

## 🧠 Part 3: The agent, v2

Three additions, each a few lines. Find the `NEW` markers:

1. **`ALLOWED`** — a set of chat ids. Empty means open to all (last week). You will fill it in Part 4.
2. **`PENDING` + `execute_calls`** — when the model reaches for a tool in `RISKY`, the loop *pauses*, the bot asks you on your phone, and `resume` picks up where it left off when you answer. Session 4's `HUMAN_IN_THE_LOOP` was a Python `input()`. This is the same gate, in the place where the human actually is.
3. **`save_memory` / `load_memory`** — after every answer, every conversation is written to `memory.json`. Stop the cell, restart it, and the bot still remembers.

In [ ]:
# ============================================
# THE AGENT, v2 — three things it couldn't do last week
# ============================================
SYSTEM = (
    "You are a personal assistant living inside a Telegram chat on the user's phone. "
    "Keep answers short and skimmable — it is a phone screen. Use tools whenever they improve accuracy. "
    "If a tool returns an error or says BLOCKED, say so honestly, never invent a result, and never retry a blocked action. "
    "If a job tool reports a FALLBACK source, say plainly that the listings are remote, not local. "
    "If get_weather's matched_place is not the city the user asked about, point that out before giving numbers."
)
MAX_STEPS, MAX_TURNS = 8, 12
AUDIT_LOG = []
MEMORY_FILE = "memory.json"

# ── NEW 1 · Who are you? ─────────────────────────────────────────────────────
ALLOWED = set()        # empty = anyone can use the bot (last week). Text /whoami, then put YOUR chat_id here: {5551234567}

# ── NEW 2 · Ask before acting ────────────────────────────────────────────────
PENDING = {}           # chat_id → the tool call waiting for a YES
YES_WORDS = {"yes", "y", "yes.", "approve", "approved", "ok", "do it"}

# ── NEW 3 · Memory that survives ─────────────────────────────────────────────
def save_memory():
    json.dump({str(k): [c.model_dump(mode="json", exclude_none=True) for c in v] for k, v in MEMORY.items()},
              open(MEMORY_FILE, "w"))

def load_memory():
    try:
        raw = json.load(open(MEMORY_FILE))
    except FileNotFoundError:
        return {}
    return {(int(k) if k.lstrip("-").isdigit() else k): [types.Content.model_validate(c) for c in v] for k, v in raw.items()}

MEMORY = load_memory()
print(f"🧠 memory loaded: {len(MEMORY)} conversation(s) on disk")


def call_model_with_retry(contents, config, max_retries=4):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except errors.APIError as e:
            if e.code in (429, 500, 503) and attempt < max_retries - 1:
                wait = 5 * (2 ** attempt); print(f"   ⏳ API returned {e.code} — waiting {wait}s"); time.sleep(wait)
            else:
                raise

def trim(history):
    starts = [i for i, c in enumerate(history) if c.role == "user" and c.parts and c.parts[0].text is not None]
    if len(starts) > MAX_TURNS:
        del history[:starts[-MAX_TURNS]]

def run_tool(chat_id, name, args):
    t0 = time.time()
    try:
        result = FUNCTIONS[name](**args) if name in FUNCTIONS else f"Error: unknown tool '{name}'"
    except Exception as e:
        result = f"Error: tool '{name}' failed: {e}"
    AUDIT_LOG.append({"time": datetime.datetime.now().strftime("%H:%M:%S"), "chat": str(chat_id)[-4:], "tool": name,
                      "args": json.dumps(args, default=str)[:60], "result": str(result)[:60], "ms": round((time.time() - t0) * 1000)})
    return result

def execute_calls(chat_id, calls, done, start, decision=None):
    """Run the model's tool calls in order. A RISKY tool stops everything and asks the human first.
    Returns (parts, None) when all calls ran, or (None, question) when we are waiting for a YES."""
    for i in range(start, len(calls)):
        name, args = calls[i]
        if name in RISKY and not (i == start and decision is not None):      # ← the approval gate
            PENDING[chat_id] = {"calls": calls, "done": done, "i": i}
            return None, f"🚦 I'd like to run {name}({json.dumps(args, default=str)}).\nReply YES to approve — anything else cancels."
        if i == start and decision is False:
            result = "BLOCKED: the user declined. Tell them it was not done and do not retry."
            AUDIT_LOG.append({"time": datetime.datetime.now().strftime("%H:%M:%S"), "chat": str(chat_id)[-4:],
                              "tool": name, "args": "(declined)", "result": "BLOCKED", "ms": 0})
        else:
            result = run_tool(chat_id, name, args)
        done.append(types.Part.from_function_response(name=name, response={"result": result}))
    return done, None

def run_loop(chat_id):
    """Session 4's loop. The only change: execute_calls can pause the loop to ask a human."""
    history = MEMORY[chat_id]
    config = types.GenerateContentConfig(tools=[types.Tool(function_declarations=TOOLS)], system_instruction=SYSTEM)
    for step in range(MAX_STEPS):
        response = call_model_with_retry(history, config)
        model_content = response.candidates[0].content
        history.append(model_content)
        calls = [(p.function_call.name, dict(p.function_call.args)) for p in (model_content.parts or []) if p.function_call]
        if not calls:
            trim(history); save_memory()                      # ← survives the cell, the tab, the day
            return response.text or "(I came back empty — try rephrasing?)"
        parts, question = execute_calls(chat_id, calls, [], 0)
        if question:
            save_memory(); return question                    # ← loop paused; resumes when the human answers
        history.append(types.Content(role="user", parts=parts))
    return "⚠️ I stopped after too many steps without reaching an answer. Try asking it differently?"

def agent_reply(chat_id, text):
    MEMORY.setdefault(chat_id, []).append(types.Content(role="user", parts=[types.Part(text=text)]))
    return run_loop(chat_id)

def resume(chat_id, text):
    """The human answered the 🚦 question."""
    p = PENDING.pop(chat_id)
    parts, question = execute_calls(chat_id, p["calls"], p["done"], p["i"], decision=text.strip().lower() in YES_WORDS)
    if question:
        return question
    MEMORY[chat_id].append(types.Content(role="user", parts=parts))
    return run_loop(chat_id)

# Quick test, no phone yet — the gate fires in Colab too
print(agent_reply(0, "Email cfo@example.com with subject 'Hi' and body 'Test from my agent.'"))
print(resume(0, "no"))
MEMORY.pop(0, None); save_memory()

## 📡 Part 4: The bridge, v2 — and the door

Same bridge as last week, with three new branches: a lock check, `/whoami`, and the 🚦 answer.

**Do this in order:**
1. Run the cell. Text your bot `/whoami`. It replies with your chat id.
2. Stop the cell (■). In **Part 3**, set `ALLOWED = {your id}`. Re-run Part 3, then re-run this cell.
3. Now hand your phone to a neighbor and have *them* text your bot from *their* phone. They should hit the door.

In [ ]:
# ============================================
# THE BRIDGE, v2 — same 40 lines, plus a door, a 🚦, and /forget
# ============================================
HELP = ("👋 I'm your agent. Math, real jobs, real weather — and I ask before I act.\n"
        "/whoami shows your id · /reset clears our chat · /forget erases it from disk too.")

def send(chat_id, text):
    for i in range(0, len(text), 4000):
        tg("sendMessage", chat_id=chat_id, text=text[i:i + 4000])

def handle(msg):
    chat_id, text = msg["chat"]["id"], msg.get("text")
    if not text:
        send(chat_id, "I only read text for now 🙂"); return
    if text.startswith("/whoami"):
        send(chat_id, f"🪪 Your chat_id is {chat_id}. The owner can add it to ALLOWED."); return
    if ALLOWED and chat_id not in ALLOWED:                                   # ← NEW 1: the door
        send(chat_id, f"🔒 This is a private assistant. (Your chat_id is {chat_id}, in case the owner wants to add you.)")
        print(f"   🔒 turned away chat …{str(chat_id)[-4:]}"); return
    if text.startswith(("/start", "/help")):
        send(chat_id, HELP); return
    if text.startswith("/reset"):
        MEMORY.pop(chat_id, None); PENDING.pop(chat_id, None); save_memory(); send(chat_id, "🧹 Fresh start."); return
    if text.startswith("/forget"):                                           # ← NEW 3: the right to be forgotten
        MEMORY.pop(chat_id, None); PENDING.pop(chat_id, None); save_memory(); send(chat_id, "🗑️ Forgotten — in memory and on disk."); return
    tg("sendChatAction", chat_id=chat_id, action="typing")
    try:
        reply = resume(chat_id, text) if chat_id in PENDING else agent_reply(chat_id, text)   # ← NEW 2: answering the 🚦
    except Exception as e:
        reply = f"😵 Something went wrong on my side ({type(e).__name__}). Try again?"
    send(chat_id, reply)

def run_bot():
    print(f"📱 Listening. Open {BOT_LINK} on your phone.  (Press ■ to stop.)")
    offset, served = 0, 0
    try:
        while True:
            try:
                updates = tg("getUpdates", http_timeout=45, offset=offset, timeout=30, allowed_updates=["message"])
            except Exception as e:
                print(f"   ⏳ polling hiccup ({type(e).__name__}) — retrying in 5s"); time.sleep(5); continue
            for u in updates:
                offset = u["update_id"] + 1
                if "message" in u:
                    served += 1
                    print(f"{datetime.datetime.now():%H:%M:%S}  💬 #{served} from {u['message']['chat'].get('first_name', 'someone')}")
                    handle(u["message"])
    except KeyboardInterrupt:
        print(f"\n🛑 Bot stopped after {served} message(s). Memory is on disk — run again and it remembers.")

run_bot()

## 🧪 Part 5: Try to break it (10 minutes)

With your bot running and your id in `ALLOWED`:

1. **The door.** A neighbor texts your bot from their phone → `🔒 This is a private assistant`. Add their id to `ALLOWED` if you want to let them in.
2. **The gate.** `Email my professor at prof@example.com that I'll be 5 minutes late` → 🚦. Reply `no`. Then ask `did you send it?` — it must say no.
3. **The gate, approved.** Ask again, reply `yes`. Check the audit log afterwards: one BLOCKED row, one SENT row.
4. **Memory that survives.** Stop the cell. Run it again. Ask `what did I ask you to email earlier?`
5. **The confident wrong answer.** `What's the weather in Bostn?` — does the agent notice it got Sweden? If not, that is a system-prompt problem, and you now know where to fix it.
6. **The honest error.** `Weather in Xyzzyville` — the tool returns an error; the agent must say so, not guess.
7. `/forget` — then ask what you talked about. Nothing. On disk too. That is a GDPR feature in one line.

Then stop the bot and read the receipt:

In [ ]:
import pandas as pd
pd.DataFrame(AUDIT_LOG) if AUDIT_LOG else print("No tool calls yet — run the bot and text it first.")

## 🚚 Part 6: bot.py, v2

Same as last week: one file, zero Colab. Drop it where last week's `bot.py` was. It will create `memory.json` next to itself.

In [ ]:
BOT_PY = r'''"""bot.py v2 — your Telegram agent with a door, an approval gate, and memory that survives.

    pip install google-genai
    export GOOGLE_API_KEY=...  TELEGRAM_BOT_TOKEN=...     (optional: ADZUNA_APP_ID / ADZUNA_APP_KEY)
    python bot.py
"""
import os
from google import genai
from google.genai import types, errors
import urllib.request, urllib.parse, json, time, datetime, re

GOOGLE_API_KEY     = os.environ["GOOGLE_API_KEY"]
TELEGRAM_BOT_TOKEN = os.environ["TELEGRAM_BOT_TOKEN"]
ADZUNA_APP_ID      = os.environ.get("ADZUNA_APP_ID")
ADZUNA_APP_KEY     = os.environ.get("ADZUNA_APP_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL  = "gemini-2.5-flash"

def tg(method, http_timeout=20, **params):
    """One function for the whole Telegram Bot API: tg('sendMessage', chat_id=..., text=...)"""
    req = urllib.request.Request(f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/{method}",
                                 data=json.dumps(params).encode(),
                                 headers={"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=http_timeout))["result"]

BOT_LINK = "https://t.me/" + tg("getMe")["username"]

# ============================================
# TOOLS — the same shape as Session 4: a job description + a Python function
# ============================================

# Tool 1: the calculator you already know
calculator_tool = types.FunctionDeclaration(
    name="calculator",
    description=(
        "A precise calculator. Use it for ANY arithmetic: addition, subtraction, "
        "multiplication, division, exponentiation. Always prefer it over mental math."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"expression": {"type": "string", "description": "e.g. '(100 * 0.15) + 50'"}},
        "required": ["expression"],
    },
)

def calculator(expression: str) -> str:
    if not all(c in set("0123456789+-*/.() ") for c in expression):
        return "Error: only numbers and + - * / . ( ) are allowed."
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"


# Tool 2: a REAL tool — live remote job listings (public API, no key needed)
jobs_tool = types.FunctionDeclaration(
    name="find_jobs",
    description=(
        "Search live remote job listings by a keyword such as 'marketing', 'analyst', "
        "'finance', 'sales' or 'design'. Returns up to 5 current openings with company, "
        "salary (when listed) and a link. Use it whenever the user asks about jobs or careers."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"keyword": {"type": "string", "description": "One or two words, e.g. 'marketing'"}},
        "required": ["keyword"],
    },
)

def find_jobs(keyword: str, limit: int = 5) -> str:
    req = urllib.request.Request("https://remoteok.com/api",
                                 headers={"User-Agent": "ISOM260-student-project"})
    raw = json.load(urllib.request.urlopen(req, timeout=25))
    pool = [j for j in raw if isinstance(j, dict) and j.get("position")]
    kw = keyword.lower()
    hits = [j for j in pool if kw in j.get("position", "").lower()]                # in the title
    hits += [j for j in pool if j not in hits and kw in " ".join(j.get("tags", [])).lower()]  # in the tags
    jobs = []
    for j in hits[:limit]:
        sal = f"${j['salary_min']:,}-${j.get('salary_max', j['salary_min']):,}" if j.get("salary_min") else "not listed"
        jobs.append({"title": j["position"], "company": j.get("company", "?"),
                     "location": j.get("location") or "Remote", "salary": sal, "url": j.get("url", "")})
    if not jobs:
        return json.dumps({"result": f"No live listings match '{keyword}'. Suggest a broader keyword."})
    return json.dumps(jobs)


# Tool 3: LOCAL jobs via Adzuna (optional key) — falls back to the remote board, and says so
adzuna_tool = types.FunctionDeclaration(
    name="find_local_jobs",
    description=(
        "Search real job postings near a US city (default Boston) by keyword, e.g. 'business analyst'. "
        "Returns up to 5 openings with company, location, salary and a link, plus a 'source' field. "
        "If source says FALLBACK, the listings are REMOTE jobs, not local — tell the user that plainly."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"keyword":  {"type": "string", "description": "Job title or skill, e.g. 'financial analyst'"},
                       "location": {"type": "string", "description": "City, e.g. 'Boston'. Default Boston."}},
        "required": ["keyword"],
    },
)

def find_local_jobs(keyword: str, location: str = "Boston", limit: int = 5) -> str:
    def fallback(why):
        return json.dumps({"source": f"FALLBACK ({why}) — these are REMOTE listings, not jobs in {location}",
                           "jobs": json.loads(find_jobs(keyword, limit))})
    if not (ADZUNA_APP_ID and ADZUNA_APP_KEY):
        return fallback("no Adzuna key")
    try:
        q = urllib.parse.urlencode({"app_id": ADZUNA_APP_ID, "app_key": ADZUNA_APP_KEY, "what": keyword,
                                    "where": location, "results_per_page": limit, "content-type": "application/json"})
        req = urllib.request.Request(f"https://api.adzuna.com/v1/api/jobs/us/search/1?{q}",
                                     headers={"User-Agent": "ISOM260-student-project"})
        raw = json.load(urllib.request.urlopen(req, timeout=20))
        jobs = []
        for r in raw.get("results", [])[:limit]:
            sal = f"${int(r['salary_min']):,}-${int(r.get('salary_max') or r['salary_min']):,}" if r.get("salary_min") else "not listed"
            jobs.append({"title": r.get("title", "?"), "company": (r.get("company") or {}).get("display_name", "?"),
                         "location": (r.get("location") or {}).get("display_name", location), "salary": sal,
                         "url": r.get("redirect_url", ""), "snippet": " ".join((r.get("description") or "").split())[:200]})
        return json.dumps({"source": f"Adzuna live · {raw.get('count', len(jobs))} postings match near {location}", "jobs": jobs})
    except Exception as e:
        return fallback(f"Adzuna failed: {type(e).__name__}")


# Tool 4: YOUR HW #5 tool goes here (paste the declaration + function, then register it below)


TOOLS     = [calculator_tool, jobs_tool, adzuna_tool]                     # ← add your tool's declaration
FUNCTIONS = {"calculator": calculator, "find_jobs": find_jobs, "find_local_jobs": find_local_jobs}   # ← and its function


# ============================================
# NEW TODAY — two more tools
# ============================================

# Tool 5: a REAL-WORLD API with real failure modes — weather, no key (Open-Meteo)
weather_tool = types.FunctionDeclaration(
    name="get_weather",
    description=(
        "Current weather and a 2-day outlook for a city. Returns the place the service actually matched "
        "(check it — a misspelled city can match the wrong country), temperatures in °F, wind, and rain chance."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"city": {"type": "string", "description": "City name, e.g. 'Boston'"}},
        "required": ["city"],
    },
)

def get_weather(city: str) -> str:
    q = urllib.parse.urlencode({"name": city, "count": 1, "language": "en"})
    geo = json.load(urllib.request.urlopen(f"https://geocoding-api.open-meteo.com/v1/search?{q}", timeout=15))
    if not geo.get("results"):
        return f"Error: no place called '{city}' was found. Ask the user to check the spelling."
    g = geo["results"][0]
    q2 = urllib.parse.urlencode({"latitude": g["latitude"], "longitude": g["longitude"],
                                 "current": "temperature_2m,wind_speed_10m", "temperature_unit": "fahrenheit",
                                 "wind_speed_unit": "mph", "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max",
                                 "forecast_days": 2, "timezone": "auto"})
    w = json.load(urllib.request.urlopen(f"https://api.open-meteo.com/v1/forecast?{q2}", timeout=15))
    return json.dumps({"matched_place": f"{g['name']}, {g.get('admin1', '')}, {g['country']}",
                       "now_f": w["current"]["temperature_2m"], "wind_mph": w["current"]["wind_speed_10m"],
                       "today_high_f": w["daily"]["temperature_2m_max"][0], "today_low_f": w["daily"]["temperature_2m_min"][0],
                       "rain_chance_pct": w["daily"]["precipitation_probability_max"][0],
                       "tomorrow_high_f": w["daily"]["temperature_2m_max"][1]})


# Tool 6: a tool that ACTS — simulated, but the agent doesn't know that
email_tool = types.FunctionDeclaration(
    name="send_email",
    description="Send an email on the user's behalf. Use ONLY when the user explicitly asks to send or email something.",
    parameters_json_schema={
        "type": "object",
        "properties": {"to": {"type": "string"}, "subject": {"type": "string"}, "body": {"type": "string"}},
        "required": ["to", "subject", "body"],
    },
)

def send_email(to: str, subject: str, body: str) -> str:
    return f"SENT (simulated) to {to} · subject '{subject}' · {len(body)} characters"


TOOLS     += [weather_tool, email_tool]
FUNCTIONS.update({"get_weather": get_weather, "send_email": send_email})
RISKY = {"send_email"}        # ← tools that must not run without a human saying yes

# ============================================
# THE AGENT, v2 — three things it couldn't do last week
# ============================================
SYSTEM = (
    "You are a personal assistant living inside a Telegram chat on the user's phone. "
    "Keep answers short and skimmable — it is a phone screen. Use tools whenever they improve accuracy. "
    "If a tool returns an error or says BLOCKED, say so honestly, never invent a result, and never retry a blocked action. "
    "If a job tool reports a FALLBACK source, say plainly that the listings are remote, not local. "
    "If get_weather's matched_place is not the city the user asked about, point that out before giving numbers."
)
MAX_STEPS, MAX_TURNS = 8, 12
AUDIT_LOG = []
MEMORY_FILE = "memory.json"

# ── NEW 1 · Who are you? ─────────────────────────────────────────────────────
ALLOWED = set()        # empty = anyone can use the bot (last week). Text /whoami, then put YOUR chat_id here: {5551234567}

# ── NEW 2 · Ask before acting ────────────────────────────────────────────────
PENDING = {}           # chat_id → the tool call waiting for a YES
YES_WORDS = {"yes", "y", "yes.", "approve", "approved", "ok", "do it"}

# ── NEW 3 · Memory that survives ─────────────────────────────────────────────
def save_memory():
    json.dump({str(k): [c.model_dump(mode="json", exclude_none=True) for c in v] for k, v in MEMORY.items()},
              open(MEMORY_FILE, "w"))

def load_memory():
    try:
        raw = json.load(open(MEMORY_FILE))
    except FileNotFoundError:
        return {}
    return {(int(k) if k.lstrip("-").isdigit() else k): [types.Content.model_validate(c) for c in v] for k, v in raw.items()}

MEMORY = load_memory()
print(f"🧠 memory loaded: {len(MEMORY)} conversation(s) on disk")


def call_model_with_retry(contents, config, max_retries=4):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except errors.APIError as e:
            if e.code in (429, 500, 503) and attempt < max_retries - 1:
                wait = 5 * (2 ** attempt); print(f"   ⏳ API returned {e.code} — waiting {wait}s"); time.sleep(wait)
            else:
                raise

def trim(history):
    starts = [i for i, c in enumerate(history) if c.role == "user" and c.parts and c.parts[0].text is not None]
    if len(starts) > MAX_TURNS:
        del history[:starts[-MAX_TURNS]]

def run_tool(chat_id, name, args):
    t0 = time.time()
    try:
        result = FUNCTIONS[name](**args) if name in FUNCTIONS else f"Error: unknown tool '{name}'"
    except Exception as e:
        result = f"Error: tool '{name}' failed: {e}"
    AUDIT_LOG.append({"time": datetime.datetime.now().strftime("%H:%M:%S"), "chat": str(chat_id)[-4:], "tool": name,
                      "args": json.dumps(args, default=str)[:60], "result": str(result)[:60], "ms": round((time.time() - t0) * 1000)})
    return result

def execute_calls(chat_id, calls, done, start, decision=None):
    """Run the model's tool calls in order. A RISKY tool stops everything and asks the human first.
    Returns (parts, None) when all calls ran, or (None, question) when we are waiting for a YES."""
    for i in range(start, len(calls)):
        name, args = calls[i]
        if name in RISKY and not (i == start and decision is not None):      # ← the approval gate
            PENDING[chat_id] = {"calls": calls, "done": done, "i": i}
            return None, f"🚦 I'd like to run {name}({json.dumps(args, default=str)}).\nReply YES to approve — anything else cancels."
        if i == start and decision is False:
            result = "BLOCKED: the user declined. Tell them it was not done and do not retry."
            AUDIT_LOG.append({"time": datetime.datetime.now().strftime("%H:%M:%S"), "chat": str(chat_id)[-4:],
                              "tool": name, "args": "(declined)", "result": "BLOCKED", "ms": 0})
        else:
            result = run_tool(chat_id, name, args)
        done.append(types.Part.from_function_response(name=name, response={"result": result}))
    return done, None

def run_loop(chat_id):
    """Session 4's loop. The only change: execute_calls can pause the loop to ask a human."""
    history = MEMORY[chat_id]
    config = types.GenerateContentConfig(tools=[types.Tool(function_declarations=TOOLS)], system_instruction=SYSTEM)
    for step in range(MAX_STEPS):
        response = call_model_with_retry(history, config)
        model_content = response.candidates[0].content
        history.append(model_content)
        calls = [(p.function_call.name, dict(p.function_call.args)) for p in (model_content.parts or []) if p.function_call]
        if not calls:
            trim(history); save_memory()                      # ← survives the cell, the tab, the day
            return response.text or "(I came back empty — try rephrasing?)"
        parts, question = execute_calls(chat_id, calls, [], 0)
        if question:
            save_memory(); return question                    # ← loop paused; resumes when the human answers
        history.append(types.Content(role="user", parts=parts))
    return "⚠️ I stopped after too many steps without reaching an answer. Try asking it differently?"

def agent_reply(chat_id, text):
    MEMORY.setdefault(chat_id, []).append(types.Content(role="user", parts=[types.Part(text=text)]))
    return run_loop(chat_id)

def resume(chat_id, text):
    """The human answered the 🚦 question."""
    p = PENDING.pop(chat_id)
    parts, question = execute_calls(chat_id, p["calls"], p["done"], p["i"], decision=text.strip().lower() in YES_WORDS)
    if question:
        return question
    MEMORY[chat_id].append(types.Content(role="user", parts=parts))
    return run_loop(chat_id)

# ============================================
# THE BRIDGE, v2 — same 40 lines, plus a door, a 🚦, and /forget
# ============================================
HELP = ("👋 I'm your agent. Math, real jobs, real weather — and I ask before I act.\n"
        "/whoami shows your id · /reset clears our chat · /forget erases it from disk too.")

def send(chat_id, text):
    for i in range(0, len(text), 4000):
        tg("sendMessage", chat_id=chat_id, text=text[i:i + 4000])

def handle(msg):
    chat_id, text = msg["chat"]["id"], msg.get("text")
    if not text:
        send(chat_id, "I only read text for now 🙂"); return
    if text.startswith("/whoami"):
        send(chat_id, f"🪪 Your chat_id is {chat_id}. The owner can add it to ALLOWED."); return
    if ALLOWED and chat_id not in ALLOWED:                                   # ← NEW 1: the door
        send(chat_id, f"🔒 This is a private assistant. (Your chat_id is {chat_id}, in case the owner wants to add you.)")
        print(f"   🔒 turned away chat …{str(chat_id)[-4:]}"); return
    if text.startswith(("/start", "/help")):
        send(chat_id, HELP); return
    if text.startswith("/reset"):
        MEMORY.pop(chat_id, None); PENDING.pop(chat_id, None); save_memory(); send(chat_id, "🧹 Fresh start."); return
    if text.startswith("/forget"):                                           # ← NEW 3: the right to be forgotten
        MEMORY.pop(chat_id, None); PENDING.pop(chat_id, None); save_memory(); send(chat_id, "🗑️ Forgotten — in memory and on disk."); return
    tg("sendChatAction", chat_id=chat_id, action="typing")
    try:
        reply = resume(chat_id, text) if chat_id in PENDING else agent_reply(chat_id, text)   # ← NEW 2: answering the 🚦
    except Exception as e:
        reply = f"😵 Something went wrong on my side ({type(e).__name__}). Try again?"
    send(chat_id, reply)

def run_bot():
    print(f"📱 Listening. Open {BOT_LINK} on your phone.  (Press ■ to stop.)")
    offset, served = 0, 0
    try:
        while True:
            try:
                updates = tg("getUpdates", http_timeout=45, offset=offset, timeout=30, allowed_updates=["message"])
            except Exception as e:
                print(f"   ⏳ polling hiccup ({type(e).__name__}) — retrying in 5s"); time.sleep(5); continue
            for u in updates:
                offset = u["update_id"] + 1
                if "message" in u:
                    served += 1
                    print(f"{datetime.datetime.now():%H:%M:%S}  💬 #{served} from {u['message']['chat'].get('first_name', 'someone')}")
                    handle(u["message"])
    except KeyboardInterrupt:
        print(f"\n🛑 Bot stopped after {served} message(s). Memory is on disk — run again and it remembers.")

if __name__ == "__main__":
    run_bot()
'''

with open("bot.py", "w") as f:
    f.write(BOT_PY)
from google.colab import files
files.download("bot.py")
print("⬇️  bot.py v2 downloaded —", len(BOT_PY.splitlines()), "lines.")

## 🎓 The four questions

You now know, from the inside, the four questions to ask any AI product a vendor shows you — or any agent you build for a company:

1. **Who can use it?** (the door — and who maintains the list)
2. **What can it do without asking?** (the gate — and who gets the 🚦)
3. **What does it remember, and can it forget?** (persistence — and the privacy bill that comes with it)
4. **What happens when the API is wrong?** (not *down* — *wrong*. Båstnäs, Sweden.)

**Boardroom sentence:** *"Our agent has an allow-list, an approval gate on every action that touches the outside world, a memory we can erase on request, and an audit log. Here is the table."*